In [24]:
##### Random Forest #####
# Say we have a decision tree, however we have a pretty complex dataset that we cannot simply take the best feature and split as it leads to inconclusive results
# How does random forest solve this? Firstly, it implements multiple decision trees. So it is more resource demanding
# Secondly, it uses vote majority. Say we have 3 features and 5 decision trees. They make the predictions : 1 4 6 7 10
#                                                                                                           2 4 6 7 10
#                                                                                                           so on so forth. now, how do we choose the best outcome for each feature? simple, we look at what value appears the most, also known as majority vote!
# Thirdly, random forests doesnt simply look at all features each time. It chooses a random number of features for EACH decision tree and based on what was chosen
# it can determine the best feature to find the best split and best threshold.
# this entire approach delivers better results thanks to the use of randomness and multiple decision trees, hence the name random forest!

import numpy as np
import pandas as pd
import kagglehub

data = None
x, y = None, None

def load_data():
    """
    Preprocessing data function
    """
    global data
    global x, y

    data = pd.read_csv("/kaggle/input/datasets/organizations/uciml/iris/Iris.csv")
    data["Species"] = data["Species"].replace({"Iris-setosa" : 1, "Iris-versicolor" : 2, "Iris-virginica" : 3})
    y = np.array(data["Species"])
    data = data.drop(columns=["Species", "Id"])
    x = np.array(data)

class Node():
    """
    Helper Node class to distinguish between split node and leaf node
    """
    def __init__(self, left=None, right=None, threshold=None, feature=None, value=None):
        self.left = left
        self.right = right
        self.threshold = threshold
        self.feature = feature
        self.value = value

class DecisionTree():
    """
    Random Forest implements at its core a multitude of decision trees, so the code is identical to
    the one for Decision Tree. I am aware that it should also be in parallel, however I did not
    implement this feature
    """
    def __init__(self, max_depth=10, min_sample_size=2, n_features=None):
        self.root = None
        self.depth = max_depth
        self.m_s_s = min_sample_size
        self.max_features = n_features # this is the first change random forest has over a regular decision tree. to have more randomness, we choose at most sqrt(max_features) random features to work with

    def _gini(self, y):
        unique, count = np.unique(y, return_counts=True)

        if(len(unique) == 0):
            return 0

        probs = count/np.sum(count)

        return 1 - np.sum(probs**2)

    def _split(self, x, threshold):
        left_indices = np.where(x < threshold)[0]
        right_indices = np.where(x >=threshold)[0]

        return left_indices, right_indices

    def _majority_class(self, y):
        unique, counts = np.unique(y, return_counts=True)
        return unique[np.argmax(counts)]

    def _candidate_thresholds(self, x):
        candidates = np.unique(x)
        for i in range(len(candidates) - 1):
            candidates[i] = (candidates[i] + candidates[i+1])/2

        return candidates[:-1]

    def _best_split(self, x, y):
        score = float('inf')
        l, r = None, None
        t, f = None, None
        m = x.shape[1]
        feature_indices = np.random.choice(m, self.max_features, replace=False)
        n = x.shape[0]
        for feature in feature_indices:
            candidates = self._candidate_thresholds(x[:, feature])
            for threshold in candidates:
                left, right = self._split(x[:, feature], threshold)
                if len(left) == 0 or len(right) == 0:
                    continue

                weighted_gini = len(left)/n * self._gini(y[left]) + len(right)/n * self._gini(y[right])

                if weighted_gini < score:
                    score = weighted_gini
                    l, r = left, right
                    t, f = threshold, feature

        return (l, r, t, f)

    def _grow_tree(self, x, y, depth=0):
        if len(np.unique(y)) == 1:
            return Node(value=y[0])
        
        if depth >= self.depth or len(y) < self.m_s_s:
            return Node(value=self._majority_class(y))

        l, r, t, f = self._best_split(x, y)
        if f is None:
            return Node(value=self._majority_class(y))

        node = Node(threshold=t, feature=f)
        node.left = self._grow_tree(x[l], y[l], depth + 1)
        node.right = self._grow_tree(x[r], y[r], depth + 1)

        return node

    def fit(self, x, y):
        if self.max_features is None:
            self.max_features = int(np.sqrt(x.shape[1]))
            
        self.root = self._grow_tree(x, y)

    def _traverse_tree(self, sample, node):
        if node.value is not None:
            return node.value

        if sample[node.feature] < node.threshold:
            return self._traverse_tree(sample, node.left)

        return self._traverse_tree(sample, node.right)

    def predict(self, x):
        preds = []
        m = x.shape[0]
        for idx in range(m):
            sample = x[idx]
            preds.append(self._traverse_tree(sample, self.root))

        return np.array(preds)

class RandomForest():
    """
    Random Forest class. Takes multitude of Decision Trees and applies majority voting system for final
    outcome
    """
    def __init__(self, n_trees=5, max_depth=10, m_s_s=2, features=2):
        self.size = n_trees
        self.depth = max_depth
        self.min_ss = m_s_s
        self.nb_f = features
        self.trees = []

    def _majority_class(self, y):
        unique, counts = np.unique(y, return_counts=True)
        return unique[np.argmax(counts)]
        
    def _bootstrap_sample(self, x, y): # the magic behind random forest is that it implements randomness ( it chooses random examples from our dataset )
        n_samples = x.shape[0]
        indices = np.random.choice(n_samples, size=n_samples, replace=True)

        return x[indices], y[indices]

    def fit(self, x, y):
        self.trees = []

        for _ in range(self.size): # the other "novelty" is that it has multiple decision trees, not just a singular one, hence the name random forest!
            tree = DecisionTree(max_depth=self.depth, min_sample_size=self.min_ss)

            x_sample, y_sample = self._bootstrap_sample(x, y)
            tree.fit(x_sample, y_sample)

            self.trees.append(tree)

    def predict(self, x): # this is another crucial aspect, random forest uses majority vote; since we have x trees and they choose m examples, we can use the transpose of their predictions and apply majority vote to get our final prediction!
        tree_preds = []

        for tree in self.trees:
            preds = tree.predict(x)
            tree_preds.append(preds)

        tree_preds = np.array(tree_preds)
        final_preds = []
        
        for sample_preds in tree_preds.T:
            final_preds.append(self._majority_class(sample_preds))

        return np.array(final_preds)
        
        
load_data()
np.random.seed(42)

indices = np.arange(len(x))
np.random.shuffle(indices)

test_size = int(0.2 * len(x))
test_idx = indices[:test_size]
train_idx = indices[test_size:]

X_train, X_test = x[train_idx], x[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

print(x.shape, y.shape)


forest = RandomForest(n_trees=20, max_depth=3)
forest.fit(X_train, y_train)

forest_preds = forest.predict(X_test)
forest_accuracy = np.mean(forest_preds == y_test)

print("Forest preds:", forest_preds)
print("True:        ", y_test)
print("Forest accuracy:", forest_accuracy)

/tmp/ipykernel_58/1002505706.py:13: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data["Species"] = data["Species"].replace({"Iris-setosa" : 1, "Iris-versicolor" : 2, "Iris-virginica" : 3})


(150, 4) (150,)
Forest preds: [2 1 3 2 2 1 2 3 2 2 3 1 1 1 1 2 3 2 2 3 1 3 1 3 3 3 3 3 1 1]
True:         [2 1 3 2 2 1 2 3 2 2 3 1 1 1 1 2 3 2 2 3 1 3 1 3 3 3 3 3 1 1]
Forest accuracy: 1.0
